<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.6-ml-trading-research/notebooks/06_ml_trading_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!printf "Y\n\n" | env BROWSER=echo stdbuf -oL -eL gh auth login \
  --hostname github.com \
  --git-protocol https \
  --web



! First copy your one-time code: 4AFD-66C1
Open this URL to continue in your web browser: https://github.com/login/device
✓ Authentication complete.
- gh config set -h github.com git_protocol https
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as Rohil121


# Bharat Portfolio Lab v0.6

## Machine Learning and Trading Research

### Objective

Test whether machine-learning signals can improve the risk-adjusted performance of an Indian equity portfolio after transaction costs, turnover and strict out-of-sample validation.

The initial research universe will be the India 10 portfolio, while the final production modules will support arbitrary eligible Indian listed equities.

## Fixed v0.6 Scope

### Prediction tasks

1. Predict each stock’s forward 21-trading-day return.
2. Estimate the probability of a positive forward return.
3. Estimate the probability of outperforming the Nifty 50 over the same period.

### Candidate features

- 1-month, 3-month, 6-month and 12-month momentum
- 21-day and 63-day realised volatility
- Recent drawdown and distance from rolling high
- Moving-average trend indicators
- Relative strength versus the Nifty 50
- Rolling beta and benchmark correlation
- Nifty 50 trend, volatility and market regime
- Lagged stock returns
- Volume-based indicators where reliable

### Candidate models

- Historical-mean baseline
- Momentum baseline
- Linear regression
- Ridge and Lasso regression
- Logistic regression
- Random forest
- Gradient boosting

### Trading strategies

- Top-ranked long-only portfolio
- Probability-weighted portfolio
- ML signal with inverse-volatility sizing
- Regime-aware ML portfolio
- Equal-weight and India 10 benchmarks

### Evaluation principles

- Expanding-window walk-forward validation
- No random train-test split
- No future information in model features
- One-day signal execution lag
- Monthly rebalancing
- Transaction costs and turnover
- CAGR, volatility, Sharpe ratio and maximum drawdown
- Hit rate, prediction error and rank information coefficient
- Performance comparison across market regimes

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


# Project folders
REPO_ROOT = Path("/content/bharat-portfolio-lab")

PROCESSED_DATA_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "ml_trading"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "ml_trading"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Research assumptions
RANDOM_SEED = 42
TRADING_DAYS_PER_YEAR = 252
FORWARD_HORIZON_DAYS = 21
MINIMUM_TRAINING_DAYS = 756

RISK_FREE_RATE = 0.065
ONE_WAY_TRANSACTION_COST = 0.0015

BENCHMARK_TICKER = "^NSEI"
BENCHMARK_NAME = "Nifty 50"


# India 10 research universe
INDIA_10_TICKERS = [
    "HDFCBANK.NS",
    "TCS.NS",
    "HINDUNILVR.NS",
    "SUNPHARMA.NS",
    "POWERGRID.NS",
    "BHARTIARTL.NS",
    "LT.NS",
    "M&M.NS",
    "BEL.NS",
    "TRENT.NS",
]

np.random.seed(
    RANDOM_SEED
)

print("v0.6 research configuration initialised.")
print("Research universe:", len(INDIA_10_TICKERS), "stocks")
print("Prediction horizon:", FORWARD_HORIZON_DAYS, "trading days")
print("Minimum training history:", MINIMUM_TRAINING_DAYS, "trading days")
print("Benchmark:", BENCHMARK_NAME)
print("Transaction cost:", f"{ONE_WAY_TRANSACTION_COST:.2%}")

v0.6 research configuration initialised.
Research universe: 10 stocks
Prediction horizon: 21 trading days
Minimum training history: 756 trading days
Benchmark: Nifty 50
Transaction cost: 0.15%


## Market Data Collection

Download daily adjusted prices and trading volumes for the India 10 stocks and the Nifty 50 benchmark.

The dataset begins in 2015 to provide enough history for:

- 12-month momentum features
- Three-year minimum training windows
- Expanding-window validation
- Multiple market regimes
- Strict out-of-sample testing

In [3]:
import yfinance as yf

DATA_START_DATE = "2015-01-01"
DATA_END_DATE = "2026-07-31"

ALL_TICKERS = (
    INDIA_10_TICKERS
    + [BENCHMARK_TICKER]
)

raw_market_data = yf.download(
    tickers=ALL_TICKERS,
    start=DATA_START_DATE,
    end=DATA_END_DATE,
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)

if raw_market_data.empty:
    raise RuntimeError(
        "No market data were downloaded."
    )

if not isinstance(
    raw_market_data.columns,
    pd.MultiIndex,
):
    raise RuntimeError(
        "Unexpected yFinance column format."
    )


# Extract adjusted close prices and volumes
close_prices = (
    raw_market_data["Close"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)

trading_volume = (
    raw_market_data["Volume"]
    .reindex(
        columns=ALL_TICKERS
    )
    .sort_index()
)


# Remove dates on which every security is missing
close_prices = close_prices.dropna(
    how="all"
)

trading_volume = trading_volume.reindex(
    close_prices.index
)


# Create a data-quality summary
quality_summary = pd.DataFrame(
    {
        "First Valid Date": (
            close_prices.apply(
                lambda series: series.first_valid_index()
            )
        ),
        "Last Valid Date": (
            close_prices.apply(
                lambda series: series.last_valid_index()
            )
        ),
        "Price Observations": (
            close_prices.notna().sum()
        ),
        "Missing Prices": (
            close_prices.isna().sum()
        ),
        "Volume Observations": (
            trading_volume.notna().sum()
        ),
    }
)

quality_summary[
    "Price Coverage"
] = (
    quality_summary[
        "Price Observations"
    ]
    / len(close_prices)
)

quality_summary[
    "Sufficient for ML"
] = (
    quality_summary[
        "Price Observations"
    ]
    >= (
        MINIMUM_TRAINING_DAYS
        + 252
    )
)


print("ML MARKET DATA CHECK")
print("=" * 65)
print(
    "Downloaded period:",
    close_prices.index.min().date(),
    "to",
    close_prices.index.max().date(),
)
print(
    "Trading dates:",
    len(close_prices),
)
print(
    "India 10 stocks:",
    len(INDIA_10_TICKERS),
)
print(
    "Benchmark included:",
    BENCHMARK_TICKER in close_prices.columns,
)
print(
    "All securities sufficient for ML:",
    quality_summary[
        "Sufficient for ML"
    ].all(),
)

display(
    quality_summary
)

ML MARKET DATA CHECK
Downloaded period: 2015-01-01 to 2026-07-30
Trading dates: 2861
India 10 stocks: 10
Benchmark included: True
All securities sufficient for ML: True


,First Valid Date,Last Valid Date,Price Observations,Missing Prices,Volume Observations,Price Coverage,Sufficient for ML
Ticker,,,,,,,
HDFCBANK.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
TCS.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
HINDUNILVR.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
SUNPHARMA.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
POWERGRID.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BHARTIARTL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
LT.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
M&M.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True
BEL.NS,2015-01-01,2026-07-30,2861,0,2861,1.000000,True


## Leakage-Safe Feature and Target Dataset

Each observation represents one stock on one trading date.

Features use only information available on or before that date. The prediction targets measure:

- Forward 21-trading-day stock return
- Whether the forward return is positive
- Forward excess return versus the Nifty 50
- Whether the stock outperforms the Nifty 50

The final trading signal will be executed with a one-day lag during backtesting.

In [5]:
# Align all securities to valid Nifty 50 trading dates

original_date_count = len(close_prices)

valid_benchmark_dates = (
    close_prices[
        BENCHMARK_TICKER
    ].notna()
)

removed_dates = (
    close_prices.index[
        ~valid_benchmark_dates
    ]
)

close_prices = (
    close_prices
    .loc[
        valid_benchmark_dates
    ]
    .copy()
)

trading_volume = (
    trading_volume
    .reindex(
        close_prices.index
    )
    .copy()
)

assert close_prices[
    BENCHMARK_TICKER
].notna().all()

assert close_prices[
    INDIA_10_TICKERS
].notna().all().all()

print("TRADING CALENDAR ALIGNMENT")
print("=" * 65)
print("Original dates:", original_date_count)
print("Benchmark-valid dates:", len(close_prices))
print("Dates removed:", len(removed_dates))
print(
    "Aligned period:",
    close_prices.index.min().date(),
    "to",
    close_prices.index.max().date(),
)
print("Benchmark missing values:", int(
    close_prices[
        BENCHMARK_TICKER
    ].isna().sum()
))

TRADING CALENDAR ALIGNMENT
Original dates: 2861
Benchmark-valid dates: 2849
Dates removed: 12
Aligned period: 2015-01-02 to 2026-07-30
Benchmark missing values: 0


In [7]:
# Daily return series
stock_daily_returns = (
    close_prices[
        INDIA_10_TICKERS
    ]
    .pct_change(
        fill_method=None
    )
)

benchmark_price = (
    close_prices[
        BENCHMARK_TICKER
    ]
)

benchmark_daily_return = (
    benchmark_price
    .pct_change(
        fill_method=None
    )
)


# Benchmark-wide features repeated for every stock
benchmark_features = pd.DataFrame(
    index=close_prices.index
)

benchmark_features[
    "benchmark_return_1d"
] = benchmark_daily_return

benchmark_features[
    "benchmark_momentum_21d"
] = (
    benchmark_price
    .pct_change(
        21,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_63d"
] = (
    benchmark_price
    .pct_change(
        63,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_momentum_126d"
] = (
    benchmark_price
    .pct_change(
        126,
        fill_method=None
    )
)

benchmark_features[
    "benchmark_volatility_21d"
] = (
    benchmark_daily_return
    .rolling(
        21
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_volatility_63d"
] = (
    benchmark_daily_return
    .rolling(
        63
    )
    .std()
    * np.sqrt(
        TRADING_DAYS_PER_YEAR
    )
)

benchmark_features[
    "benchmark_drawdown_252d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        252
    )
    .max()
    - 1
)

benchmark_features[
    "benchmark_ma_gap_200d"
] = (
    benchmark_price
    / benchmark_price
    .rolling(
        200
    )
    .mean()
    - 1
)

benchmark_features[
    "benchmark_bull_regime"
] = (
    benchmark_price
    > benchmark_price
    .rolling(
        200
    )
    .mean()
).astype(float)


# Forward benchmark target
benchmark_forward_return_21d = (
    benchmark_price.shift(
        -FORWARD_HORIZON_DAYS
    )
    / benchmark_price
    - 1
)


# Build one feature frame per stock
stock_feature_frames = {}

for ticker in INDIA_10_TICKERS:

    stock_price = (
        close_prices[
            ticker
        ]
    )

    stock_return = (
        stock_daily_returns[
            ticker
        ]
    )

    stock_volume = (
        trading_volume[
            ticker
        ]
    )

    stock_frame = pd.DataFrame(
        index=close_prices.index
    )

    # Recent returns and momentum
    stock_frame[
        "return_1d"
    ] = stock_return

    stock_frame[
        "return_5d"
    ] = (
        stock_price
        .pct_change(
            5,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_21d"
    ] = (
        stock_price
        .pct_change(
            21,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_63d"
    ] = (
        stock_price
        .pct_change(
            63,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_126d"
    ] = (
        stock_price
        .pct_change(
            126,
            fill_method=None
        )
    )

    stock_frame[
        "momentum_252d"
    ] = (
        stock_price
        .pct_change(
            252,
            fill_method=None
        )
    )

    # Risk and drawdown
    stock_frame[
        "volatility_21d"
    ] = (
        stock_return
        .rolling(
            21
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "volatility_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .std()
        * np.sqrt(
            TRADING_DAYS_PER_YEAR
        )
    )

    stock_frame[
        "drawdown_252d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            252
        )
        .max()
        - 1
    )

    # Trend indicators
    stock_frame[
        "ma_gap_21d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_63d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            63
        )
        .mean()
        - 1
    )

    stock_frame[
        "ma_gap_200d"
    ] = (
        stock_price
        / stock_price
        .rolling(
            200
        )
        .mean()
        - 1
    )

    # Relative strength versus Nifty 50
    stock_frame[
        "relative_strength_63d"
    ] = (
        stock_frame[
            "momentum_63d"
        ]
        - benchmark_features[
            "benchmark_momentum_63d"
        ]
    )

    stock_frame[
        "relative_strength_126d"
    ] = (
        stock_frame[
            "momentum_126d"
        ]
        - benchmark_features[
            "benchmark_momentum_126d"
        ]
    )

    # Rolling market sensitivity
    rolling_covariance = (
        stock_return
        .rolling(
            63
        )
        .cov(
            benchmark_daily_return
        )
    )

    rolling_benchmark_variance = (
        benchmark_daily_return
        .rolling(
            63
        )
        .var()
    )

    stock_frame[
        "beta_63d"
    ] = (
        rolling_covariance
        / rolling_benchmark_variance
    )

    stock_frame[
        "benchmark_correlation_63d"
    ] = (
        stock_return
        .rolling(
            63
        )
        .corr(
            benchmark_daily_return
        )
    )

    # Volume features
    stock_frame[
        "volume_ratio_21d"
    ] = (
        stock_volume
        / stock_volume
        .rolling(
            21
        )
        .mean()
        - 1
    )

    stock_frame[
        "volume_trend_21_63d"
    ] = (
        stock_volume
        .rolling(
            21
        )
        .mean()
        / stock_volume
        .rolling(
            63
        )
        .mean()
        - 1
    )

    # Add market-wide features
    stock_frame = stock_frame.join(
        benchmark_features
    )

    # Forward targets
    forward_return = (
        stock_price.shift(
            -FORWARD_HORIZON_DAYS
        )
        / stock_price
        - 1
    )

    forward_excess_return = (
        forward_return
        - benchmark_forward_return_21d
    )

    stock_frame[
        "forward_return_21d"
    ] = forward_return

    stock_frame[
        "forward_excess_return_21d"
    ] = forward_excess_return

    stock_frame[
        "positive_return_target"
    ] = (
        forward_return
        .gt(0)
        .where(
            forward_return.notna()
        )
        .astype(float)
    )

    stock_frame[
        "outperform_target"
    ] = (
        forward_excess_return
        .gt(0)
        .where(
            forward_excess_return.notna()
        )
        .astype(float)
    )

    stock_feature_frames[
        ticker
    ] = stock_frame


# Convert into one stock-date panel
ml_panel_raw = (
    pd.concat(
        stock_feature_frames,
        names=[
            "Ticker",
            "Date",
        ],
    )
    .reset_index()
)


feature_columns = [
    "return_1d",
    "return_5d",
    "momentum_21d",
    "momentum_63d",
    "momentum_126d",
    "momentum_252d",
    "volatility_21d",
    "volatility_63d",
    "drawdown_252d",
    "ma_gap_21d",
    "ma_gap_63d",
    "ma_gap_200d",
    "relative_strength_63d",
    "relative_strength_126d",
    "beta_63d",
    "benchmark_correlation_63d",
    "volume_ratio_21d",
    "volume_trend_21_63d",
    "benchmark_return_1d",
    "benchmark_momentum_21d",
    "benchmark_momentum_63d",
    "benchmark_momentum_126d",
    "benchmark_volatility_21d",
    "benchmark_volatility_63d",
    "benchmark_drawdown_252d",
    "benchmark_ma_gap_200d",
    "benchmark_bull_regime",
]

target_columns = [
    "forward_return_21d",
    "forward_excess_return_21d",
    "positive_return_target",
    "outperform_target",
]


# Retain only observations with complete features and targets
ml_panel = (
    ml_panel_raw
    .dropna(
        subset=(
            feature_columns
            + target_columns
        )
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# Store classification targets as integers
ml_panel[
    "positive_return_target"
] = (
    ml_panel[
        "positive_return_target"
    ]
    .astype(int)
)

ml_panel[
    "outperform_target"
] = (
    ml_panel[
        "outperform_target"
    ]
    .astype(int)
)


# Save a reproducible research dataset
feature_dataset_path = (
    PROCESSED_DATA_DIR
    / "india10_ml_feature_panel.csv"
)

ml_panel.to_csv(
    feature_dataset_path,
    index=False,
)


# Validation
assert ml_panel[
    feature_columns
].notna().all().all()

assert ml_panel[
    target_columns
].notna().all().all()

assert set(
    ml_panel[
        "Ticker"
    ].unique()
) == set(
    INDIA_10_TICKERS
)

assert (
    ml_panel[
        "Date"
    ].max()
    <= close_prices.index[
        -FORWARD_HORIZON_DAYS - 1
    ]
)


print("ML FEATURE DATASET CHECK")
print("=" * 65)
print(
    "Panel observations:",
    f"{len(ml_panel):,}",
)
print(
    "Stocks:",
    ml_panel[
        "Ticker"
    ].nunique(),
)
print(
    "Features:",
    len(feature_columns),
)
print(
    "Dataset period:",
    ml_panel[
        "Date"
    ].min().date(),
    "to",
    ml_panel[
        "Date"
    ].max().date(),
)
print(
    "Positive-return rate:",
    f"{ml_panel['positive_return_target'].mean():.2%}",
)
print(
    "Nifty outperformance rate:",
    f"{ml_panel['outperform_target'].mean():.2%}",
)
print(
    "Missing feature values:",
    int(
        ml_panel[
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
)
print(
    "Saved dataset:",
    feature_dataset_path,
)

display(
    ml_panel.head()
)

ML FEATURE DATASET CHECK
Panel observations: 25,760
Stocks: 10
Features: 27
Dataset period: 2016-01-14 to 2026-07-01
Positive-return rate: 58.69%
Nifty outperformance rate: 52.03%
Missing feature values: 0
Saved dataset: /content/bharat-portfolio-lab/data/processed/ml_trading/india10_ml_feature_panel.csv


,Ticker,Date,return_1d,return_5d,momentum_21d,momentum_63d,momentum_126d,momentum_252d,volatility_21d,volatility_63d,...,benchmark_momentum_126d,benchmark_volatility_21d,benchmark_volatility_63d,benchmark_drawdown_252d,benchmark_ma_gap_200d,benchmark_bull_regime,forward_return_21d,forward_excess_return_21d,positive_return_target,outperform_target
0,BEL.NS,2016-01-14,-0.023923,0.010744,0.102567,0.082936,0.151348,0.351668,0.287186,0.259007,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.110123,-0.060519,0,0
1,BHARTIARTL.NS,2016-01-14,0.000486,-0.042190,0.008492,-0.111766,-0.284446,-0.149490,0.283601,0.250530,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,0.032875,0.082478,1,1
2,HDFCBANK.NS,2016-01-14,-0.009810,-0.006107,-0.005024,-0.033513,-0.022625,0.095659,0.122117,0.128907,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.072541,-0.022938,0,0
3,HINDUNILVR.NS,2016-01-14,-0.006604,0.008534,-0.014005,0.019752,-0.094833,0.115090,0.180885,0.181416,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,-0.027259,0.022344,0,1
4,LT.NS,2016-01-14,-0.019105,-0.059475,-0.109659,-0.266264,-0.380025,-0.253073,0.167221,0.192795,...,-0.095065,0.148547,0.121527,-0.162229,-0.075498,0.0,0.012999,0.062603,1,1


In [8]:
expected_last_target_date = (
    close_prices.index[
        -FORWARD_HORIZON_DAYS - 1
    ]
)

actual_last_panel_date = (
    ml_panel[
        "Date"
    ].max()
)

calendar_gap_days = (
    expected_last_target_date
    - actual_last_panel_date
).days

assert calendar_gap_days >= 0
assert calendar_gap_days <= 7

print("CORRECTED FEATURE PANEL")
print("=" * 65)
print(
    "Dataset period:",
    ml_panel["Date"].min().date(),
    "to",
    actual_last_panel_date.date(),
)
print(
    "Expected final eligible date:",
    expected_last_target_date.date(),
)
print(
    "Panel observations:",
    f"{len(ml_panel):,}",
)
print(
    "Features:",
    len(feature_columns),
)
print(
    "Missing feature values:",
    int(
        ml_panel[
            feature_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
)
print("Calendar correction:", "PASSED")

CORRECTED FEATURE PANEL
Dataset period: 2016-01-14 to 2026-07-01
Expected final eligible date: 2026-07-01
Panel observations: 25,760
Features: 27
Missing feature values: 0
Calendar correction: PASSED


## Feature Quality and Target Analysis

Before model training, evaluate:

- Missing, infinite and constant feature values
- Duplicate stock-date observations
- Forward-return and classification-target distributions
- Target behaviour across calendar years
- Cross-sectional predictive relationship between each feature and future excess returns

The daily rank Information Coefficient measures whether a feature correctly ranks stocks from weaker to stronger future Nifty 50 outperformance. It is an exploratory statistic, not evidence of a profitable strategy.

In [10]:
# =========================================================
# CORRECT CROSS-SECTIONAL IC VALIDATION
# Separate stock-ranking features from market-wide features
# =========================================================

# Number of valid daily cross-sectional IC observations
daily_ic_valid_counts = (
    daily_feature_rank_ic
    .notna()
    .sum()
)

cross_sectional_features = (
    daily_ic_valid_counts[
        daily_ic_valid_counts > 0
    ]
    .index
    .tolist()
)

non_cross_sectional_features = (
    daily_ic_valid_counts[
        daily_ic_valid_counts == 0
    ]
    .index
    .tolist()
)

expected_market_wide_features = [
    feature
    for feature in feature_columns
    if feature.startswith(
        "benchmark_"
    )
]


# Market-wide features should be the only variables
# without a valid cross-sectional rank IC.
unexpected_non_cross_sectional_features = sorted(
    set(
        non_cross_sectional_features
    )
    - set(
        expected_market_wide_features
    )
)

if unexpected_non_cross_sectional_features:
    raise RuntimeError(
        "Unexpected features have no valid daily rank IC:\n"
        + "\n".join(
            unexpected_non_cross_sectional_features
        )
    )


# Label each feature according to its research role
feature_quality_summary[
    "Feature Role"
] = [
    (
        "Market-wide regime feature"
        if feature in expected_market_wide_features
        else "Cross-sectional stock feature"
    )
    for feature in feature_quality_summary.index
]

feature_quality_summary[
    "Valid Daily IC Observations"
] = (
    daily_ic_valid_counts
    .reindex(
        feature_quality_summary.index
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


# Sort stock-ranking features by predictive rank strength.
# Market-wide features remain in the table but appear afterward.
feature_quality_summary[
    "IC Available"
] = (
    feature_quality_summary[
        "Valid Daily IC Observations"
    ]
    > 0
)

feature_quality_summary = (
    feature_quality_summary
    .sort_values(
        by=[
            "IC Available",
            "Absolute Mean Daily Rank IC",
        ],
        ascending=[
            False,
            False,
        ],
        na_position="last",
    )
)


# Re-save the corrected feature-quality output
feature_quality_summary.to_csv(
    feature_quality_path
)


# Correct validation
assert duplicate_stock_dates == 0
assert infinite_feature_values == 0
assert missing_feature_values == 0
assert not constant_features
assert not feature_target_overlap

assert len(
    cross_sectional_features
) > 0

assert not (
    unexpected_non_cross_sectional_features
)

assert set(
    non_cross_sectional_features
).issubset(
    set(
        expected_market_wide_features
    )
)


top_rank_features = (
    feature_quality_summary.loc[
        feature_quality_summary[
            "IC Available"
        ],
        [
            "Feature Role",
            "Mean Daily Rank IC",
            "Positive Daily IC Rate",
            "Pooled Spearman IC",
            "Rank IC Information Ratio",
            "Valid Daily IC Observations",
        ],
    ]
    .head(
        10
    )
)


print("CORRECTED FEATURE QUALITY CHECK")
print("=" * 70)
print(
    "Total features:",
    len(
        feature_columns
    ),
)
print(
    "Cross-sectional stock features:",
    len(
        cross_sectional_features
    ),
)
print(
    "Market-wide features:",
    len(
        non_cross_sectional_features
    ),
)
print(
    "Market-wide feature names:",
    ", ".join(
        non_cross_sectional_features
    ),
)
print(
    "Unexpected features without rank IC:",
    len(
        unexpected_non_cross_sectional_features
    ),
)
print(
    "Duplicate stock-date rows:",
    duplicate_stock_dates,
)
print(
    "Missing feature values:",
    missing_feature_values,
)
print(
    "Infinite feature values:",
    infinite_feature_values,
)
print(
    "Feature-quality validation:",
    "PASSED",
)

print("\nTOP CROSS-SECTIONAL FEATURES")
display(
    top_rank_features
)

print("\nTARGET SUMMARY BY YEAR")
display(
    target_summary_by_year
)

CORRECTED FEATURE QUALITY CHECK
Total features: 27
Cross-sectional stock features: 18
Market-wide features: 9
Market-wide feature names: benchmark_return_1d, benchmark_momentum_21d, benchmark_momentum_63d, benchmark_momentum_126d, benchmark_volatility_21d, benchmark_volatility_63d, benchmark_drawdown_252d, benchmark_ma_gap_200d, benchmark_bull_regime
Unexpected features without rank IC: 0
Duplicate stock-date rows: 0
Missing feature values: 0
Infinite feature values: 0
Feature-quality validation: PASSED

TOP CROSS-SECTIONAL FEATURES


,Feature Role,Mean Daily Rank IC,Positive Daily IC Rate,Pooled Spearman IC,Rank IC Information Ratio,Valid Daily IC Observations
momentum_21d,Cross-sectional stock feature,-0.056686,0.427407,-0.063482,-0.155586,2576
ma_gap_21d,Cross-sectional stock feature,-0.048414,0.440217,-0.057060,-0.135389,2576
momentum_252d,Cross-sectional stock feature,0.046217,0.533773,0.057143,0.114503,2576
ma_gap_63d,Cross-sectional stock feature,-0.039596,0.473602,-0.041242,-0.105522,2576
return_5d,Cross-sectional stock feature,-0.038392,0.459627,-0.040758,-0.111295,2576
volatility_63d,Cross-sectional stock feature,0.027692,0.528727,-0.023106,0.070869,2576
drawdown_252d,Cross-sectional stock feature,-0.020416,0.485637,0.009491,-0.052889,2576
volume_ratio_21d,Cross-sectional stock feature,-0.018474,0.490683,-0.011486,-0.054417,2575
return_1d,Cross-sectional stock feature,-0.018182,0.476320,-0.019014,-0.052734,2575
momentum_63d,Cross-sectional stock feature,-0.014130,0.486413,-0.014523,-0.037171,2576



TARGET SUMMARY BY YEAR


,Observations,Trading_Dates,Mean_Forward_Return,Median_Forward_Return,Mean_Forward_Excess_Return,Positive_Return_Rate,Nifty_Outperformance_Rate
Year,,,,,,,
2016,2360,236,0.013168,0.012921,0.000882,0.583051,0.486441
2017,2480,248,0.026994,0.025243,0.005202,0.675000,0.547984
2018,2450,245,-0.002761,0.000579,-0.003702,0.502449,0.503673
2019,2410,241,0.015713,0.010745,0.004464,0.564315,0.523237
2020,2500,250,0.026631,0.024852,0.007840,0.605600,0.462400
2021,2480,248,0.026382,0.020346,0.007128,0.620565,0.493145
2022,2480,248,0.015402,0.010850,0.013211,0.550806,0.560887
2023,2450,245,0.032624,0.027344,0.016027,0.683265,0.604082
2024,2460,246,0.027401,0.019790,0.020307,0.613821,0.578862


## Leakage-Safe Baseline Signals

Before training machine-learning models, evaluate simple investment signals:

1. Expanding historical mean excess return by stock
2. Six-month momentum
3. Twelve-month momentum
4. Blended six- and twelve-month momentum
5. One-month reversal
6. Equal-weight India 10 benchmark

Signals are evaluated on monthly rebalance dates. Historical target information is lagged by 21 trading days so that only outcomes already observable at the signal date are used.

These results measure ranking ability and forward returns. They are not yet a complete transaction-cost-adjusted portfolio backtest.

In [11]:
# =========================================================
# LEAKAGE-SAFE BASELINE SIGNAL EVALUATION
# =========================================================

baseline_panel = ml_panel.copy()

baseline_panel[
    "Date"
] = pd.to_datetime(
    baseline_panel[
        "Date"
    ]
)

market_calendar = pd.DatetimeIndex(
    close_prices.index
).sort_values()

eligible_dates = pd.DatetimeIndex(
    sorted(
        baseline_panel[
            "Date"
        ].unique()
    )
)


# ---------------------------------------------------------
# 1. Select the final eligible observation in each month
# ---------------------------------------------------------

monthly_rebalance_dates = (
    pd.Series(
        eligible_dates,
        index=eligible_dates,
    )
    .groupby(
        eligible_dates.to_period(
            "M"
        )
    )
    .max()
    .tolist()
)


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def cross_sectional_zscore(
    values: pd.Series,
) -> pd.Series:

    standard_deviation = values.std(
        ddof=0
    )

    if (
        pd.isna(
            standard_deviation
        )
        or standard_deviation == 0
    ):
        return pd.Series(
            0.0,
            index=values.index,
        )

    return (
        values
        - values.mean()
    ) / standard_deviation


def calculate_rank_ic(
    frame: pd.DataFrame,
    score_column: str,
) -> float:

    if (
        frame[
            score_column
        ].nunique()
        <= 1
    ):
        return np.nan

    return frame[
        [
            score_column,
            "forward_excess_return_21d",
        ]
    ].corr(
        method="spearman"
    ).iloc[
        0,
        1,
    ]


# ---------------------------------------------------------
# 3. Generate monthly out-of-sample baseline predictions
# ---------------------------------------------------------

baseline_prediction_records = []

for rebalance_date in monthly_rebalance_dates:

    calendar_position = (
        market_calendar.get_indexer(
            [
                rebalance_date
            ]
        )[0]
    )

    if calendar_position == -1:
        continue

    # Require at least three years of prior market history.
    if calendar_position < (
        MINIMUM_TRAINING_DAYS
        + FORWARD_HORIZON_DAYS
    ):
        continue

    target_availability_position = (
        calendar_position
        - FORWARD_HORIZON_DAYS
    )

    target_availability_date = (
        market_calendar[
            target_availability_position
        ]
    )

    training_panel = (
        baseline_panel.loc[
            baseline_panel[
                "Date"
            ]
            <= target_availability_date
        ]
        .copy()
    )

    current_cross_section = (
        baseline_panel.loc[
            baseline_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            current_cross_section
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if training_panel.empty:
        continue

    historical_stock_means = (
        training_panel
        .groupby(
            "Ticker"
        )[
            "forward_excess_return_21d"
        ]
        .mean()
    )

    global_historical_mean = (
        training_panel[
            "forward_excess_return_21d"
        ]
        .mean()
    )

    current_cross_section[
        "historical_mean_score"
    ] = (
        current_cross_section[
            "Ticker"
        ]
        .map(
            historical_stock_means
        )
        .fillna(
            global_historical_mean
        )
    )

    current_cross_section[
        "momentum_126d_score"
    ] = (
        current_cross_section[
            "momentum_126d"
        ]
    )

    current_cross_section[
        "momentum_252d_score"
    ] = (
        current_cross_section[
            "momentum_252d"
        ]
    )

    current_cross_section[
        "blended_momentum_score"
    ] = (
        0.50
        * cross_sectional_zscore(
            current_cross_section[
                "momentum_126d"
            ]
        )
        + 0.50
        * cross_sectional_zscore(
            current_cross_section[
                "momentum_252d"
            ]
        )
    )

    current_cross_section[
        "short_term_reversal_score"
    ] = (
        -current_cross_section[
            "momentum_21d"
        ]
    )

    current_cross_section[
        "Training Cutoff"
    ] = target_availability_date

    baseline_prediction_records.append(
        current_cross_section
    )


if not baseline_prediction_records:
    raise RuntimeError(
        "No valid monthly baseline predictions were generated."
    )

baseline_predictions = (
    pd.concat(
        baseline_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Evaluate each baseline signal
# ---------------------------------------------------------

baseline_score_columns = {
    "Historical Mean":
        "historical_mean_score",

    "6-Month Momentum":
        "momentum_126d_score",

    "12-Month Momentum":
        "momentum_252d_score",

    "Blended Momentum":
        "blended_momentum_score",

    "1-Month Reversal":
        "short_term_reversal_score",
}

TOP_STOCK_COUNT = 3

baseline_evaluation_records = []

for baseline_name, score_column in (
    baseline_score_columns.items()
):

    daily_rank_ic_values = []
    selected_return_values = []
    selected_excess_values = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        rebalance_date,
        rebalance_cross_section,
    ) in baseline_predictions.groupby(
        "Date"
    ):

        rank_ic = calculate_rank_ic(
            frame=rebalance_cross_section,
            score_column=score_column,
        )

        daily_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            rebalance_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                score_column,
            )
        )

        selected_return_values.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_values.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = pd.Series(
        daily_rank_ic_values,
        dtype=float,
    ).dropna()

    baseline_evaluation_records.append(
        {
            "Baseline":
                baseline_name,

            "Rebalances":
                baseline_predictions[
                    "Date"
                ].nunique(),

            "Selected Stocks":
                TOP_STOCK_COUNT,

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Mean Selected Forward Return":
                np.mean(
                    selected_return_values
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_values
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


# ---------------------------------------------------------
# 5. Add the equal-weight India 10 benchmark
# ---------------------------------------------------------

equal_weight_monthly_results = (
    baseline_predictions
    .groupby(
        "Date"
    )
    .agg(
        Mean_Forward_Return=(
            "forward_return_21d",
            "mean",
        ),

        Mean_Forward_Excess_Return=(
            "forward_excess_return_21d",
            "mean",
        ),

        Positive_Return_Rate=(
            "positive_return_target",
            "mean",
        ),

        Outperformance_Rate=(
            "outperform_target",
            "mean",
        ),
    )
)

baseline_evaluation_records.append(
    {
        "Baseline":
            "Equal-Weight India 10",

        "Rebalances":
            len(
                equal_weight_monthly_results
            ),

        "Selected Stocks":
            len(
                INDIA_10_TICKERS
            ),

        "Mean Rank IC":
            np.nan,

        "Median Rank IC":
            np.nan,

        "Positive Rank IC Rate":
            np.nan,

        "Mean Selected Forward Return":
            equal_weight_monthly_results[
                "Mean_Forward_Return"
            ].mean(),

        "Mean Selected Excess Return":
            equal_weight_monthly_results[
                "Mean_Forward_Excess_Return"
            ].mean(),

        "Selected Positive Return Rate":
            equal_weight_monthly_results[
                "Positive_Return_Rate"
            ].mean(),

        "Selected Outperformance Rate":
            equal_weight_monthly_results[
                "Outperformance_Rate"
            ].mean(),
    }
)


baseline_evaluation = (
    pd.DataFrame(
        baseline_evaluation_records
    )
    .set_index(
        "Baseline"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Save baseline research datasets
# ---------------------------------------------------------

baseline_predictions_path = (
    PROCESSED_DATA_DIR
    / "monthly_baseline_predictions.csv"
)

baseline_evaluation_path = (
    PROCESSED_DATA_DIR
    / "baseline_signal_evaluation.csv"
)

baseline_predictions.to_csv(
    baseline_predictions_path,
    index=False,
)

baseline_evaluation.to_csv(
    baseline_evaluation_path
)


# ---------------------------------------------------------
# 7. Leakage and quality validation
# ---------------------------------------------------------

assert (
    baseline_predictions[
        "Training Cutoff"
    ]
    < baseline_predictions[
        "Date"
    ]
).all()

assert (
    baseline_predictions[
        "Date"
    ].nunique()
    >= 60
)

assert (
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert (
    baseline_predictions[
        list(
            baseline_score_columns.values()
        )
    ]
    .notna()
    .all()
    .all()
)

assert len(
    baseline_evaluation
) == 6


# ---------------------------------------------------------
# 8. Results
# ---------------------------------------------------------

display_evaluation = (
    baseline_evaluation.copy()
)

percentage_columns = [
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_evaluation[
        column
    ] = (
        display_evaluation[
            column
        ]
        * 100
    )


print("BASELINE SIGNAL EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    baseline_predictions[
        "Date"
    ].min().date(),
    "to",
    baseline_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    baseline_predictions[
        "Date"
    ].nunique(),
)
print(
    "Stocks per rebalance:",
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Ticker"
    ]
    .nunique()
    .median(),
)
print(
    "Historical target lag:",
    FORWARD_HORIZON_DAYS,
    "trading days",
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Leakage-safe training cutoffs:",
    "PASSED",
)
print(
    "Baseline evaluation:",
    "PASSED",
)

display(
    display_evaluation.round(
        2
    )
)

BASELINE SIGNAL EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Stocks per rebalance: 10.0
Historical target lag: 21 trading days
Top stocks selected: 3
Leakage-safe training cutoffs: PASSED
Baseline evaluation: PASSED


,Rebalances,Selected Stocks,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Baseline,,,,,,,,,
1-Month Reversal,101,3,6.89,6.67,59.41,2.80,1.79,59.41,55.12
12-Month Momentum,101,3,3.04,5.45,58.42,2.23,1.21,59.74,51.16
Historical Mean,101,3,3.41,6.67,56.44,2.18,1.17,58.42,53.80
6-Month Momentum,101,3,2.66,5.45,59.41,2.05,1.03,57.43,53.14
Blended Momentum,101,3,3.77,3.03,55.45,1.94,0.92,58.09,52.15
Equal-Weight India 10,101,10,NaN,NaN,NaN,1.80,0.79,58.12,51.58


## Walk-Forward Linear Regression Models

The first machine-learning models predict each stock’s forward 21-trading-day excess return over the Nifty 50.

Models:

- Ordinary Least Squares Linear Regression
- Ridge Regression

Both models use:

- An expanding historical training window
- Only targets observable before the prediction date
- Training-only feature standardisation
- Monthly out-of-sample predictions
- The same evaluation dates as the baseline strategies

These are deliberately simple, untuned models. More complex tree-based models will only be considered after establishing whether linear relationships contain useful predictive information.

In [12]:
# =========================================================
# WALK-FORWARD LINEAR REGRESSION MODELS
# =========================================================

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Prepare the modelling panel and rebalance schedule
# ---------------------------------------------------------

regression_panel = ml_panel.copy()

regression_panel[
    "Date"
] = pd.to_datetime(
    regression_panel[
        "Date"
    ]
)

rebalance_schedule = (
    baseline_predictions
    .groupby(
        "Date"
    )[
        "Training Cutoff"
    ]
    .first()
    .sort_index()
)

rebalance_schedule.index = (
    pd.to_datetime(
        rebalance_schedule.index
    )
)

rebalance_schedule = pd.to_datetime(
    rebalance_schedule
)


# ---------------------------------------------------------
# 2. Define simple regression models
# ---------------------------------------------------------

regression_models = {
    "Linear Regression":
        Pipeline(
            steps=[
                (
                    "standard_scaler",
                    StandardScaler(),
                ),
                (
                    "regression_model",
                    LinearRegression(),
                ),
            ]
        ),

    "Ridge Regression":
        Pipeline(
            steps=[
                (
                    "standard_scaler",
                    StandardScaler(),
                ),
                (
                    "regression_model",
                    Ridge(
                        alpha=10.0
                    ),
                ),
            ]
        ),
}


# ---------------------------------------------------------
# 3. Generate monthly walk-forward predictions
# ---------------------------------------------------------

regression_prediction_records = []

for (
    rebalance_date,
    training_cutoff,
) in rebalance_schedule.items():

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if (
        len(
            training_sample
        )
        < (
            len(
                INDIA_10_TICKERS
            )
            * 252
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(
            float
        )
    )

    y_train = (
        training_sample[
            "forward_excess_return_21d"
        ]
        .astype(
            float
        )
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(
            float
        )
    )

    for (
        model_name,
        model_pipeline,
    ) in regression_models.items():

        model_pipeline.fit(
            X_train,
            y_train,
        )

        predicted_excess_return = (
            model_pipeline.predict(
                X_predict
            )
        )

        model_predictions = (
            prediction_sample[
                [
                    "Date",
                    "Ticker",
                    "forward_return_21d",
                    "forward_excess_return_21d",
                    "positive_return_target",
                    "outperform_target",
                ]
            ]
            .copy()
        )

        model_predictions[
            "Model"
        ] = model_name

        model_predictions[
            "Predicted Excess Return"
        ] = predicted_excess_return

        model_predictions[
            "Training Cutoff"
        ] = training_cutoff

        model_predictions[
            "Training Observations"
        ] = len(
            training_sample
        )

        regression_prediction_records.append(
            model_predictions
        )


if not regression_prediction_records:
    raise RuntimeError(
        "No walk-forward regression predictions "
        "were generated."
    )

regression_predictions = (
    pd.concat(
        regression_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Model",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Evaluate each model
# ---------------------------------------------------------

TOP_STOCK_COUNT = 3

regression_evaluation_records = []

for (
    model_name,
    model_predictions,
) in regression_predictions.groupby(
    "Model"
):

    actual_values = (
        model_predictions[
            "forward_excess_return_21d"
        ]
    )

    predicted_values = (
        model_predictions[
            "Predicted Excess Return"
        ]
    )

    monthly_rank_ic_values = []
    selected_forward_returns = []
    selected_excess_returns = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        prediction_date,
        prediction_cross_section,
    ) in model_predictions.groupby(
        "Date"
    ):

        if (
            prediction_cross_section[
                "Predicted Excess Return"
            ].nunique()
            > 1
        ):

            rank_ic = (
                prediction_cross_section[
                    [
                        "Predicted Excess Return",
                        "forward_excess_return_21d",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ]
            )

        else:

            rank_ic = np.nan

        monthly_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            prediction_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                "Predicted Excess Return",
            )
        )

        selected_forward_returns.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_returns.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = (
        pd.Series(
            monthly_rank_ic_values,
            dtype=float,
        )
        .dropna()
    )

    directional_accuracy = (
        (
            predicted_values
            > 0
        )
        == (
            actual_values
            > 0
        )
    ).mean()

    regression_evaluation_records.append(
        {
            "Model":
                model_name,

            "Rebalances":
                model_predictions[
                    "Date"
                ].nunique(),

            "Predictions":
                len(
                    model_predictions
                ),

            "Mean Absolute Error":
                mean_absolute_error(
                    actual_values,
                    predicted_values,
                ),

            "Root Mean Squared Error":
                np.sqrt(
                    mean_squared_error(
                        actual_values,
                        predicted_values,
                    )
                ),

            "Pooled R-Squared":
                r2_score(
                    actual_values,
                    predicted_values,
                ),

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Directional Accuracy":
                directional_accuracy,

            "Mean Selected Forward Return":
                np.mean(
                    selected_forward_returns
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_returns
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


regression_evaluation = (
    pd.DataFrame(
        regression_evaluation_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Create comparison with the strongest simple baselines
# ---------------------------------------------------------

regression_baseline_comparison = (
    regression_evaluation[
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ]
    ]
    .copy()
)

for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
    "Equal-Weight India 10",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    regression_baseline_comparison.loc[
        baseline_name
    ] = {
        "Rebalances":
            baseline_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            baseline_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            baseline_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            baseline_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            baseline_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            baseline_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            baseline_row[
                "Selected Outperformance Rate"
            ],
    }


regression_baseline_comparison = (
    regression_baseline_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 6. Save research outputs
# ---------------------------------------------------------

regression_predictions_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_regression_predictions.csv"
)

regression_evaluation_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_regression_evaluation.csv"
)

regression_comparison_path = (
    PROCESSED_DATA_DIR
    / "regression_baseline_comparison.csv"
)

regression_predictions.to_csv(
    regression_predictions_path,
    index=False,
)

regression_evaluation.to_csv(
    regression_evaluation_path
)

regression_baseline_comparison.to_csv(
    regression_comparison_path
)


# ---------------------------------------------------------
# 7. Leakage and data-quality validation
# ---------------------------------------------------------

assert (
    regression_predictions[
        "Training Cutoff"
    ]
    < regression_predictions[
        "Date"
    ]
).all()

assert (
    regression_predictions[
        "Predicted Excess Return"
    ]
    .notna()
    .all()
)

assert np.isfinite(
    regression_predictions[
        "Predicted Excess Return"
    ]
).all()

assert (
    regression_predictions
    .groupby(
        [
            "Date",
            "Model",
        ]
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert (
    regression_evaluation[
        "Rebalances"
    ]
    .eq(
        regression_predictions[
            "Date"
        ].nunique()
    )
    .all()
)


# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

display_regression_evaluation = (
    regression_evaluation.copy()
)

percentage_columns = [
    "Mean Absolute Error",
    "Root Mean Squared Error",
    "Pooled R-Squared",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Directional Accuracy",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_regression_evaluation[
        column
    ] = (
        display_regression_evaluation[
            column
        ]
        * 100
    )


display_comparison = (
    regression_baseline_comparison.copy()
)

comparison_percentage_columns = [
    "Mean Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in comparison_percentage_columns:

    display_comparison[
        column
    ] = (
        display_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD REGRESSION EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    regression_predictions[
        "Date"
    ].min().date(),
    "to",
    regression_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    regression_predictions[
        "Date"
    ].nunique(),
)
print(
    "Models:",
    regression_predictions[
        "Model"
    ].nunique(),
)
print(
    "Features:",
    len(
        feature_columns
    ),
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Training-only standardisation:",
    "PASSED",
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "Regression evaluation:",
    "PASSED",
)

print("\nREGRESSION MODEL METRICS")
display(
    display_regression_evaluation.round(
        2
    )
)

print("\nREGRESSION VS SIMPLE BASELINES")
display(
    display_comparison.round(
        2
    )
)

WALK-FORWARD REGRESSION EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Models: 2
Features: 27
Top stocks selected: 3
Training-only standardisation: PASSED
Leakage-safe cutoffs: PASSED
Regression evaluation: PASSED

REGRESSION MODEL METRICS


,Rebalances,Predictions,Mean Absolute Error,Root Mean Squared Error,Pooled R-Squared,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Directional Accuracy,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,,,,,,,
Linear Regression,101,1010,5.57,7.46,-5.68,0.09,-3.03,48.51,48.91,2.33,1.31,58.75,51.82
Ridge Regression,101,1010,5.57,7.45,-5.50,0.02,-3.03,48.51,49.21,2.33,1.31,58.75,51.82



REGRESSION VS SIMPLE BASELINES


,Rebalances,Mean Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,
1-Month Reversal,101.0,6.89,59.41,2.80,1.79,59.41,55.12
Linear Regression,101.0,0.09,48.51,2.33,1.31,58.75,51.82
Ridge Regression,101.0,0.02,48.51,2.33,1.31,58.75,51.82
12-Month Momentum,101.0,3.04,58.42,2.23,1.21,59.74,51.16
Equal-Weight India 10,101.0,NaN,NaN,1.80,0.79,58.12,51.58


## Walk-Forward Logistic Classification Models

Two classification models are evaluated:

1. Positive Return Classifier  
   Estimates the probability that a stock generates a positive return over the next 21 trading days.

2. Nifty Outperformance Classifier  
   Estimates the probability that a stock outperforms the Nifty 50 over the next 21 trading days.

Both models use expanding-window training, training-only feature standardisation and monthly out-of-sample predictions.

The three stocks with the highest predicted probabilities are selected at each rebalance date.

In [13]:
# =========================================================
# WALK-FORWARD LOGISTIC CLASSIFICATION MODELS
# =========================================================

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Define the two classification tasks
# ---------------------------------------------------------

classification_tasks = {
    "Positive Return Logistic": {
        "target_column":
            "positive_return_target",

        "ranking_outcome":
            "forward_return_21d",
    },

    "Nifty Outperformance Logistic": {
        "target_column":
            "outperform_target",

        "ranking_outcome":
            "forward_excess_return_21d",
    },
}


base_logistic_pipeline = Pipeline(
    steps=[
        (
            "standard_scaler",
            StandardScaler(),
        ),
        (
            "classification_model",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=2_000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)


# ---------------------------------------------------------
# 2. Generate walk-forward monthly predictions
# ---------------------------------------------------------

classification_prediction_records = []

for (
    rebalance_date,
    training_cutoff,
) in rebalance_schedule.items():

    training_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            <= training_cutoff
        ]
        .copy()
    )

    prediction_sample = (
        regression_panel.loc[
            regression_panel[
                "Date"
            ]
            == rebalance_date
        ]
        .copy()
    )

    if (
        len(
            prediction_sample
        )
        != len(
            INDIA_10_TICKERS
        )
    ):
        continue

    if (
        len(
            training_sample
        )
        < (
            len(
                INDIA_10_TICKERS
            )
            * 252
        )
    ):
        continue

    X_train = (
        training_sample[
            feature_columns
        ]
        .astype(float)
    )

    X_predict = (
        prediction_sample[
            feature_columns
        ]
        .astype(float)
    )

    for (
        model_name,
        task_details,
    ) in classification_tasks.items():

        target_column = (
            task_details[
                "target_column"
            ]
        )

        ranking_outcome = (
            task_details[
                "ranking_outcome"
            ]
        )

        y_train = (
            training_sample[
                target_column
            ]
            .astype(int)
        )

        if y_train.nunique() != 2:
            raise RuntimeError(
                f"{model_name} training data does not "
                "contain both target classes."
            )

        fitted_model = clone(
            base_logistic_pipeline
        )

        fitted_model.fit(
            X_train,
            y_train,
        )

        predicted_probability = (
            fitted_model.predict_proba(
                X_predict
            )[
                :,
                1,
            ]
        )

        predicted_class = (
            predicted_probability
            >= 0.50
        ).astype(int)

        model_predictions = (
            prediction_sample[
                [
                    "Date",
                    "Ticker",
                    "forward_return_21d",
                    "forward_excess_return_21d",
                    "positive_return_target",
                    "outperform_target",
                ]
            ]
            .copy()
        )

        model_predictions[
            "Model"
        ] = model_name

        model_predictions[
            "Target Column"
        ] = target_column

        model_predictions[
            "Actual Class"
        ] = (
            prediction_sample[
                target_column
            ]
            .astype(int)
            .to_numpy()
        )

        model_predictions[
            "Ranking Outcome"
        ] = (
            prediction_sample[
                ranking_outcome
            ]
            .to_numpy()
        )

        model_predictions[
            "Predicted Probability"
        ] = predicted_probability

        model_predictions[
            "Predicted Class"
        ] = predicted_class

        model_predictions[
            "Training Cutoff"
        ] = training_cutoff

        model_predictions[
            "Training Observations"
        ] = len(
            training_sample
        )

        classification_prediction_records.append(
            model_predictions
        )


if not classification_prediction_records:
    raise RuntimeError(
        "No classification predictions were generated."
    )


classification_predictions = (
    pd.concat(
        classification_prediction_records,
        ignore_index=True,
    )
    .sort_values(
        [
            "Date",
            "Model",
            "Ticker",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 3. Evaluate classification and stock selection
# ---------------------------------------------------------

TOP_STOCK_COUNT = 3

classification_evaluation_records = []

for (
    model_name,
    model_predictions,
) in classification_predictions.groupby(
    "Model"
):

    actual_class = (
        model_predictions[
            "Actual Class"
        ]
        .astype(int)
    )

    predicted_class = (
        model_predictions[
            "Predicted Class"
        ]
        .astype(int)
    )

    predicted_probability = (
        model_predictions[
            "Predicted Probability"
        ]
    )

    monthly_rank_ic_values = []
    selected_forward_returns = []
    selected_excess_returns = []
    selected_positive_rates = []
    selected_outperformance_rates = []

    for (
        prediction_date,
        prediction_cross_section,
    ) in model_predictions.groupby(
        "Date"
    ):

        if (
            prediction_cross_section[
                "Predicted Probability"
            ].nunique()
            > 1
        ):

            rank_ic = (
                prediction_cross_section[
                    [
                        "Predicted Probability",
                        "Ranking Outcome",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ]
            )

        else:

            rank_ic = np.nan

        monthly_rank_ic_values.append(
            rank_ic
        )

        selected_stocks = (
            prediction_cross_section
            .nlargest(
                TOP_STOCK_COUNT,
                "Predicted Probability",
            )
        )

        selected_forward_returns.append(
            selected_stocks[
                "forward_return_21d"
            ].mean()
        )

        selected_excess_returns.append(
            selected_stocks[
                "forward_excess_return_21d"
            ].mean()
        )

        selected_positive_rates.append(
            selected_stocks[
                "positive_return_target"
            ].mean()
        )

        selected_outperformance_rates.append(
            selected_stocks[
                "outperform_target"
            ].mean()
        )

    valid_rank_ic = (
        pd.Series(
            monthly_rank_ic_values,
            dtype=float,
        )
        .dropna()
    )

    classification_evaluation_records.append(
        {
            "Model":
                model_name,

            "Rebalances":
                model_predictions[
                    "Date"
                ].nunique(),

            "Predictions":
                len(
                    model_predictions
                ),

            "Accuracy":
                accuracy_score(
                    actual_class,
                    predicted_class,
                ),

            "Balanced Accuracy":
                balanced_accuracy_score(
                    actual_class,
                    predicted_class,
                ),

            "ROC AUC":
                roc_auc_score(
                    actual_class,
                    predicted_probability,
                ),

            "Log Loss":
                log_loss(
                    actual_class,
                    predicted_probability,
                    labels=[
                        0,
                        1,
                    ],
                ),

            "Brier Score":
                brier_score_loss(
                    actual_class,
                    predicted_probability,
                ),

            "Mean Rank IC":
                valid_rank_ic.mean(),

            "Median Rank IC":
                valid_rank_ic.median(),

            "Positive Rank IC Rate":
                valid_rank_ic.gt(
                    0
                ).mean(),

            "Mean Selected Forward Return":
                np.mean(
                    selected_forward_returns
                ),

            "Mean Selected Excess Return":
                np.mean(
                    selected_excess_returns
                ),

            "Selected Positive Return Rate":
                np.mean(
                    selected_positive_rates
                ),

            "Selected Outperformance Rate":
                np.mean(
                    selected_outperformance_rates
                ),
        }
    )


classification_evaluation = (
    pd.DataFrame(
        classification_evaluation_records
    )
    .set_index(
        "Model"
    )
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 4. Compare classifiers with existing baselines
# ---------------------------------------------------------

classification_baseline_comparison = (
    classification_evaluation[
        [
            "Rebalances",
            "Mean Rank IC",
            "Positive Rank IC Rate",
            "Mean Selected Forward Return",
            "Mean Selected Excess Return",
            "Selected Positive Return Rate",
            "Selected Outperformance Rate",
        ]
    ]
    .copy()
)

for baseline_name in [
    "1-Month Reversal",
    "12-Month Momentum",
    "Equal-Weight India 10",
]:

    baseline_row = (
        baseline_evaluation.loc[
            baseline_name
        ]
    )

    classification_baseline_comparison.loc[
        baseline_name
    ] = {
        "Rebalances":
            baseline_row[
                "Rebalances"
            ],

        "Mean Rank IC":
            baseline_row[
                "Mean Rank IC"
            ],

        "Positive Rank IC Rate":
            baseline_row[
                "Positive Rank IC Rate"
            ],

        "Mean Selected Forward Return":
            baseline_row[
                "Mean Selected Forward Return"
            ],

        "Mean Selected Excess Return":
            baseline_row[
                "Mean Selected Excess Return"
            ],

        "Selected Positive Return Rate":
            baseline_row[
                "Selected Positive Return Rate"
            ],

        "Selected Outperformance Rate":
            baseline_row[
                "Selected Outperformance Rate"
            ],
    }


classification_baseline_comparison = (
    classification_baseline_comparison
    .sort_values(
        "Mean Selected Excess Return",
        ascending=False,
    )
)


# ---------------------------------------------------------
# 5. Save research outputs
# ---------------------------------------------------------

classification_predictions_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_classification_predictions.csv"
)

classification_evaluation_path = (
    PROCESSED_DATA_DIR
    / "walk_forward_classification_evaluation.csv"
)

classification_comparison_path = (
    PROCESSED_DATA_DIR
    / "classification_baseline_comparison.csv"
)

classification_predictions.to_csv(
    classification_predictions_path,
    index=False,
)

classification_evaluation.to_csv(
    classification_evaluation_path
)

classification_baseline_comparison.to_csv(
    classification_comparison_path
)


# ---------------------------------------------------------
# 6. Leakage and quality validation
# ---------------------------------------------------------

assert (
    classification_predictions[
        "Training Cutoff"
    ]
    < classification_predictions[
        "Date"
    ]
).all()

assert (
    classification_predictions[
        "Predicted Probability"
    ]
    .between(
        0,
        1,
    )
    .all()
)

assert (
    classification_predictions[
        "Predicted Probability"
    ]
    .notna()
    .all()
)

assert (
    classification_predictions
    .groupby(
        [
            "Date",
            "Model",
        ]
    )[
        "Ticker"
    ]
    .nunique()
    .eq(
        len(
            INDIA_10_TICKERS
        )
    )
    .all()
)

assert len(
    classification_evaluation
) == 2


# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

display_classification_evaluation = (
    classification_evaluation.copy()
)

percentage_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "ROC AUC",
    "Brier Score",
    "Mean Rank IC",
    "Median Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in percentage_columns:

    display_classification_evaluation[
        column
    ] = (
        display_classification_evaluation[
            column
        ]
        * 100
    )


display_classification_comparison = (
    classification_baseline_comparison.copy()
)

comparison_percentage_columns = [
    "Mean Rank IC",
    "Positive Rank IC Rate",
    "Mean Selected Forward Return",
    "Mean Selected Excess Return",
    "Selected Positive Return Rate",
    "Selected Outperformance Rate",
]

for column in comparison_percentage_columns:

    display_classification_comparison[
        column
    ] = (
        display_classification_comparison[
            column
        ]
        * 100
    )


print("WALK-FORWARD CLASSIFICATION EVALUATION")
print("=" * 70)
print(
    "Evaluation period:",
    classification_predictions[
        "Date"
    ].min().date(),
    "to",
    classification_predictions[
        "Date"
    ].max().date(),
)
print(
    "Monthly rebalances:",
    classification_predictions[
        "Date"
    ].nunique(),
)
print(
    "Classification models:",
    classification_predictions[
        "Model"
    ].nunique(),
)
print(
    "Features:",
    len(
        feature_columns
    ),
)
print(
    "Top stocks selected:",
    TOP_STOCK_COUNT,
)
print(
    "Leakage-safe cutoffs:",
    "PASSED",
)
print(
    "Classification evaluation:",
    "PASSED",
)

print("\nCLASSIFICATION MODEL METRICS")
display(
    display_classification_evaluation.round(
        2
    )
)

print("\nCLASSIFICATION VS SIMPLE BASELINES")
display(
    display_classification_comparison.round(
        2
    )
)

WALK-FORWARD CLASSIFICATION EVALUATION
Evaluation period: 2018-03-28 to 2026-07-01
Monthly rebalances: 101
Classification models: 2
Features: 27
Top stocks selected: 3
Leakage-safe cutoffs: PASSED
Classification evaluation: PASSED

CLASSIFICATION MODEL METRICS


,Rebalances,Predictions,Accuracy,Balanced Accuracy,ROC AUC,Log Loss,Brier Score,Mean Rank IC,Median Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,,,,,,,,
Positive Return Logistic,101,1010,52.28,46.63,45.51,0.73,26.57,-3.15,1.82,51.49,2.19,1.17,58.42,51.49
Nifty Outperformance Logistic,101,1010,48.61,48.33,47.43,0.73,26.37,-3.89,-6.67,45.54,1.76,0.74,57.43,48.84



CLASSIFICATION VS SIMPLE BASELINES


,Rebalances,Mean Rank IC,Positive Rank IC Rate,Mean Selected Forward Return,Mean Selected Excess Return,Selected Positive Return Rate,Selected Outperformance Rate
Model,,,,,,,
1-Month Reversal,101.0,6.89,59.41,2.80,1.79,59.41,55.12
12-Month Momentum,101.0,3.04,58.42,2.23,1.21,59.74,51.16
Positive Return Logistic,101.0,-3.15,51.49,2.19,1.17,58.42,51.49
Equal-Weight India 10,101.0,NaN,NaN,1.80,0.79,58.12,51.58
Nifty Outperformance Logistic,101.0,-3.89,45.54,1.76,0.74,57.43,48.84
